In [269]:
import pandas as pd
import random
from typing import Annotated, Final
from annotated_doc import Doc

RANDOM_SEED = 42
NUM_PROPERTIES = 100
NUM_DATES = 500
BASE_BOOST = 25
SUMMER_SEASONAL_BOOST = 75
WINTER_SEASON_BOOST = 50
EVENT_BOOST = 75
WEEKEND_BOOST = 50
HOLIDAY_BOOST = 75
BASE_PROPERTY_PRICINGS : Final[
    Annotated[
        dict[str,int],
        Doc("Base pricing of all properties.")
        ]] = { 
              "Entire house": 3000,
              "Private room": 1000,
              "Luxury suite": 2000
              }
"""Base prices for each property type"""

RATING_PRICE_FACTOR : Final[
    Annotated[
    int,
    Doc("""
        The factor by which rating of a property affects the price.
        When multiplied by the difference between the actual rating and avg. it gives
        the amount by which the price should be increased or reduced.
        """
    )
    ]] = 100

RATING_DEMAND_FACTOR = 60

In [ ]:
def getSeason(month:int) -> str:
  """
  Function to return the season (according to seasons in the Indian subcontinent) 
  based on month of the year
  """
  
  if 3 <= month <= 6:
    return "Summer"
  
  elif 7 <= month <= 9:
    return "Monsoon"
  
  else:
    return "Winter"

In [ ]:
def generatePropertyFeatures() -> dict[str, str | int | float]:
  """
  Function to generate property features for the properties
  """
  
  # Generate a random property type
  property_int = random.randint(0,2)
  property_type = ""
  
  if(property_int == 0):
    property_type = "Entire house"
  elif(property_int == 1):
    property_type = "Private room"
  else:
    property_type = "Luxury suite"
  
  # Generate a random base rating, amenities score and location score
  base_rating = random.randint(1,100)
  amenities_score = random.randint(1,10)
  location_score = random.randint(1,10)
  
  # Better aminities mean better rating so we use a
  # simple coorelation to factor in aminities score
  amenities_boost = 0
  if(amenities_score <= 3):
    amenities_boost = 30
  elif(3 < amenities_score <= 6):
    amenities_boost = 60
  else:
    amenities_boost = 90
  
  # A good location means a better rating so we use a
  # simple coorelation to factor in location score
  location_boost = 0
  if(location_score <= 3):
    location_boost = 30
  elif(3 < location_score <= 6):
    location_boost = 60
  else:
    location_boost = 90
  
  # Final rating will be out of 300 but will never be 300
  # since we only use a max boost of 90 so no rating is completely 5 star
  final_rating = base_rating + amenities_boost + location_boost
  
  final_norm_rating = (final_rating/300) * 5
  
  property_pricing = BASE_PROPERTY_PRICINGS[property_type] + (final_norm_rating - 3.5) * RATING_PRICE_FACTOR
      
  return {
    'property_type': property_type,
    'rating': round(final_norm_rating,2),
    'amenities_score': amenities_score,
    'location_score': location_score,
    'base_price': round(property_pricing,2)
  }

In [ ]:
def processDates(date: pd.Timestamp) -> dict:
  processed_date = date.strftime('%Y-%m-%d')
  month = date.month
  weekday = date.day_name()
  weekend = 1 if date.weekday() == 5 or date.weekday() == 6 else 0
  season = getSeason(date.month)
  holiday = 0 if random.randint(0,100) < 95 else 1
  nearby_event =  0 if random.randint(0,100) < 90 else 1
  
  return {
    'date': processed_date,
    'month': month,
    'weekday': weekday,
    'weekend': weekend,
    'season': season,
    'holiday': holiday,
    'nearby_event': nearby_event,
  }

In [ ]:
def generateTemporalFeatures() -> list[dict[str,int | str]]:
  """
  Get dates from Jan 1 2024 to 31 Dec 2025 along with features -
  month: int,
  weekday: str,
  weekend: 0 if weekend else 1,
  season: str (season according to the Indian Subcontinent)
  """
  date_range = pd.date_range('2024-01-01','2025-12-31').to_list()
  temporal_features = list(map(processDates,date_range))  
  
  return temporal_features

In [274]:
def processTemporalAndPropertyFeatures(index:int, property_features: dict, temporal_features: dict):
  property_id = f"P{index}"
  date = temporal_features['date']
  property_type = property_features['property_type']
  location_score = property_features['location_score']
  amenities_score = property_features['amenities_score']
  rating = property_features['rating']
  base_price = property_features['base_price']
  month = temporal_features['month']
  weekday = temporal_features['weekday']
  weekend = temporal_features['weekend']
  season = temporal_features['season']
  holiday = temporal_features['holiday']
  nearby_event = temporal_features['nearby_event']
  
  base_demand = 100 + (RATING_DEMAND_FACTOR * (rating/5))
  
  seasonal_demand_boost = 0
  if(season == "Summmer"):
    seasonal_demand_boost = SUMMER_SEASONAL_BOOST
  elif(season == "Winter"):
    seasonal_demand_boost = WINTER_SEASON_BOOST
  else:
    seasonal_demand_boost = BASE_BOOST
  
  holiday_demand_boost = 0
  if holiday:
    holiday_demand_boost = HOLIDAY_BOOST
  else:
    holiday_demand_boost = BASE_BOOST
  
  weekend_demand_boost = 0
  if weekend:
    weekend_demand_boost = WEEKEND_BOOST
  else:
    weekend_demand_boost = BASE_BOOST
  
  event_demand_boost = 0
  if nearby_event:
    event_demand_boost = EVENT_BOOST
  else:
    event_demand_boost = BASE_BOOST
  
  final_demand = round((base_demand + seasonal_demand_boost + holiday_demand_boost + weekend_demand_boost + event_demand_boost)/500,2)
  
  return {
  'property_id': property_id,
  'date': date,
  'property_type': property_type,
  'location_score': location_score,
  'amenities_score': amenities_score,
  'rating': rating,
  'base_price': base_price,
  'month': month,
  'weekday': weekday,
  'weekend': weekend,
  'season': season,
  'holiday': holiday,
  'demand': final_demand,
  'competitor_price': 0,
  'nearby_event': nearby_event,
  'market_trend': 0,
  'final_price': 0,
  'occupancy_rate': 0,
  'revenue': 0
  }


In [ ]:
dates = generateTemporalFeatures()
dates = dates[:500]

num_holidays = 0
for i in dates:
  if i['holiday'] == 1:
    num_holidays+=1

complete_dataset = []
index = 0
for i in range(NUM_PROPERTIES):
  property = generatePropertyFeatures()
  for j in range(NUM_DATES):
    complete_dataset.append(processTemporalAndPropertyFeatures(i,property,dates[j]))
    index+=1

complete_dataset_df = pd.DataFrame(complete_dataset)
print(complete_dataset_df[:10])


  property_id        date property_type  location_score  amenities_score  \
0          P0  2024-01-01  Entire house               7                3   
1          P0  2024-01-02  Entire house               7                3   
2          P0  2024-01-03  Entire house               7                3   
3          P0  2024-01-04  Entire house               7                3   
4          P0  2024-01-05  Entire house               7                3   
5          P0  2024-01-06  Entire house               7                3   
6          P0  2024-01-07  Entire house               7                3   
7          P0  2024-01-08  Entire house               7                3   
8          P0  2024-01-09  Entire house               7                3   
9          P0  2024-01-10  Entire house               7                3   

   rating  base_price  month    weekday  weekend  season  holiday  demand  \
0    2.55      2905.0      1     Monday        0  Winter        0    0.51   
1    2.55